# The Gutenberg-Richter Anomaly Detector

The **Gutenberg-Richter (GR) law** is one of the most fundamental empirical relations in seismology. It states that the frequency-magnitude distribution of earthquakes follows:

$$\log_{10}(N) = a - bM$$

where $N$ is the cumulative number of earthquakes with magnitude $\geq M$, $a$ describes the overall seismicity rate, and $b$ (the **b-value**) characterizes the relative proportion of large to small events.

For **tectonic** earthquakes, the b-value is remarkably stable at $b \approx 1.0$. However, **induced seismicity** — earthquakes triggered by wastewater injection, hydraulic fracturing, or other anthropogenic activities — systematically departs from this baseline. Induced sequences typically exhibit elevated b-values in the range $b \approx 1.2$–$1.5$, reflecting a deficit of larger-magnitude events relative to the tectonic expectation.

This notebook leverages temporal variations in the b-value as a **fingerprint** to distinguish induced from tectonic seismicity. We apply rolling b-value estimation, changepoint detection, and a novel **phase portrait** visualization to track how seismicity character evolves over time in Oklahoma (induced), the Permian Basin (emerging), and Southern California (tectonic reference).

---
## Imports

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path so we can import src modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from src.catalog import estimate_mc, REGION_COLORS
from src.gutenberg_richter import (
    estimate_b_value,
    bootstrap_b_value,
    rolling_b_value,
    detect_changepoints,
    ks_test_magnitudes,
)

# Plotting defaults
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (12, 7)

# Region colors
COLORS = {
    'oklahoma': '#E63946',
    'permian': '#F4A261',
    'socal': '#457B9D',
}

FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

print('Imports complete.')

---
## Load Data

Load processed Parquet catalogs for each region and estimate the magnitude of completeness ($M_c$) using the maximum curvature method.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

regions = {}
mc_values = {}

for name in ['oklahoma', 'permian', 'socal']:
    path = DATA_DIR / f'{name}.parquet'
    df = pd.read_parquet(path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    
    # Estimate magnitude of completeness
    mc = estimate_mc(df['mag'])
    mc_values[name] = mc
    
    # Filter to events above Mc
    df_filtered = df[df['mag'] >= mc].copy().reset_index(drop=True)
    regions[name] = df_filtered
    
    print(f"{name:>10s}: {len(df):>6d} total events | "
          f"Mc = {mc:.1f} | {len(df_filtered):>6d} events above Mc")

print('\nData loaded successfully.')

---
## Gutenberg-Richter Plots

For each region, plot the frequency-magnitude distribution (FMD) with the maximum-likelihood GR fit line. The magnitude of completeness $M_c$ is marked with a dashed vertical line.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, (name, df) in zip(axes, regions.items()):
    mc = mc_values[name]
    mags = df['mag'].values
    color = COLORS[name]
    
    # Build frequency-magnitude distribution
    mag_bins = np.arange(mc, mags.max() + 0.1, 0.1)
    # Cumulative counts: N(>=M)
    cumulative_counts = np.array([np.sum(mags >= m) for m in mag_bins])
    
    # Plot observed FMD
    ax.semilogy(mag_bins, cumulative_counts, 'o', color=color, markersize=4,
                alpha=0.7, label='Observed')
    
    # Fit GR law
    b, b_std = estimate_b_value(mags, mc)
    a = np.log10(np.sum(mags >= mc)) + b * mc
    fit_line = 10 ** (a - b * mag_bins)
    
    ax.semilogy(mag_bins, fit_line, '-', color='black', linewidth=1.5,
                label=f'GR fit: b = {b:.2f} \u00b1 {b_std:.2f}')
    
    # Mark Mc
    ax.axvline(mc, color='gray', linestyle='--', linewidth=1, alpha=0.7,
               label=f'$M_c$ = {mc:.1f}')
    
    ax.set_xlabel('Magnitude', fontsize=11)
    ax.set_title(name.replace('socal', 'Southern California')
                     .replace('oklahoma', 'Oklahoma')
                     .replace('permian', 'Permian Basin').title(),
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(mc - 0.3, mags.max() + 0.2)

axes[0].set_ylabel('Cumulative Number of Events $N(\\geq M)$', fontsize=11)

fig.suptitle('Gutenberg-Richter Frequency-Magnitude Distributions', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'gutenberg_richter_fmd.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved: figures/gutenberg_richter_fmd.png')

---
## Rolling b-values and Changepoint Detection

The b-value is not static — it evolves through time as the stress regime, fluid pressures, and fault conditions change. We compute rolling b-values using a sliding window of 200 events with a step of 50 events, then apply BIC-penalized changepoint detection to identify statistically significant shifts.

Key dates are marked:
- **2009**: Ramp-up of wastewater injection volumes in Oklahoma
- **2015**: Oklahoma Corporation Commission implements volume reduction regulations

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

rolling_results = {}

for ax, (name, df) in zip(axes, regions.items()):
    mc = mc_values[name]
    color = COLORS[name]
    
    # Compute rolling b-values
    rb = rolling_b_value(df['time'].values, df['mag'].values, mc,
                         window_size=200, step=50)
    rolling_results[name] = rb
    
    # Convert center_time to datetime
    rb['center_time'] = pd.to_datetime(rb['center_time'], utc=True)
    
    # Plot b-value time series
    ax.plot(rb['center_time'], rb['b_value'], color=color, linewidth=1.5,
            label=f'{name.title()} b-value')
    ax.fill_between(rb['center_time'],
                    rb['b_value'] - rb['b_std'],
                    rb['b_value'] + rb['b_std'],
                    alpha=0.2, color=color)
    
    # Changepoint detection
    if len(rb) > 40:
        cps = detect_changepoints(rb['b_value'].values, min_size=20, penalty_factor=3.0)
        for cp in cps:
            if cp < len(rb):
                ax.axvline(rb['center_time'].iloc[cp], color='darkred',
                           linestyle=':', linewidth=1.2, alpha=0.7)
    
    # Key dates
    for year, label_text in [(2009, 'Injection ramp-up'), (2015, 'Regulation')]:
        date = pd.Timestamp(f'{year}-01-01', tz='UTC')
        if rb['center_time'].min() < date < rb['center_time'].max():
            ax.axvline(date, color='black', linestyle='--', linewidth=1, alpha=0.5)
            ax.text(date, ax.get_ylim()[1] * 0.98, f' {label_text} ({year})',
                    fontsize=8, ha='left', va='top', style='italic')
    
    # Reference b=1.0 line
    ax.axhline(1.0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)
    
    display_name = (name.replace('socal', 'Southern California')
                        .replace('oklahoma', 'Oklahoma')
                        .replace('permian', 'Permian Basin'))
    ax.set_title(f'{display_name} — Rolling b-value (window=200, step=50)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('b-value', fontsize=11)
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Date', fontsize=11)

fig.suptitle('Temporal Evolution of b-values with Changepoint Detection',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'rolling_bvalues_changepoints.png',
            bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved: figures/rolling_bvalues_changepoints.png')

---
## b-value Phase Portrait

This is the **signature visualization** of the Seismic Fingerprints project. Instead of plotting b-value against time, we plot b-value against the logarithm of the seismicity rate, creating a **phase portrait** that reveals the dynamical character of each region's seismicity.

In this space:
- **Tectonic seismicity** (SoCal) orbits a stable attractor near $(\log_{10}(rate), b) \approx (\text{const}, 1.0)$
- **Induced seismicity** (Oklahoma) traces a characteristic **loop**: as injection ramps up, both rate and b-value increase, then as regulation takes effect, the trajectory retreats
- **The Permian Basin** can be placed on this portrait to assess where it currently sits relative to Oklahoma's historical trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for name in ['oklahoma', 'socal', 'permian']:
    rb = rolling_results[name].copy()
    color = COLORS[name]
    
    # Compute log10(rate)
    log_rate = np.log10(rb['rate'].values)
    b_vals = rb['b_value'].values
    times = pd.to_datetime(rb['center_time'], utc=True)
    
    # Compute year as float for colormap
    years = times.dt.year + times.dt.dayofyear / 365.25
    
    # Filter out NaN/inf values
    valid = np.isfinite(log_rate) & np.isfinite(b_vals)
    log_rate = log_rate[valid]
    b_vals = b_vals[valid]
    years_valid = years[valid].values
    
    # Color-code by year using viridis
    norm = plt.Normalize(years_valid.min(), years_valid.max())
    
    # Plot the trajectory as colored scatter
    sc = ax.scatter(log_rate, b_vals, c=years_valid, cmap='viridis',
                    norm=norm, s=20, alpha=0.7, zorder=3,
                    edgecolors=color, linewidths=0.5)
    
    # Add arrows showing direction of time
    n_arrows = min(8, len(log_rate) // 5)
    if n_arrows > 0:
        arrow_indices = np.linspace(0, len(log_rate) - 2, n_arrows, dtype=int)
        for idx in arrow_indices:
            dx = log_rate[idx + 1] - log_rate[idx]
            dy = b_vals[idx + 1] - b_vals[idx]
            ax.annotate('', xy=(log_rate[idx + 1], b_vals[idx + 1]),
                        xytext=(log_rate[idx], b_vals[idx]),
                        arrowprops=dict(arrowstyle='->', color=color,
                                        lw=1.5, alpha=0.6))
    
    # Label the trajectory endpoint
    display_name = (name.replace('socal', 'SoCal')
                        .replace('oklahoma', 'Oklahoma')
                        .replace('permian', 'Permian Basin'))
    ax.annotate(display_name, xy=(log_rate[-1], b_vals[-1]),
                fontsize=10, fontweight='bold', color=color,
                xytext=(10, 10), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color=color, lw=1))

# Add colorbar
cbar = plt.colorbar(sc, ax=ax, label='Year', pad=0.02)

# Reference lines
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5,
           label='Tectonic b = 1.0')
ax.axhspan(1.2, 1.5, alpha=0.05, color='red', label='Induced b-value range')

# Custom legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['oklahoma'],
           markersize=8, label='Oklahoma'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['permian'],
           markersize=8, label='Permian Basin'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORS['socal'],
           markersize=8, label='SoCal'),
    Line2D([0], [0], color='gray', linestyle='--', label='Tectonic b = 1.0'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

ax.set_xlabel('$\\log_{10}$(Seismicity Rate) [events/day]', fontsize=12)
ax.set_ylabel('b-value', fontsize=12)
ax.set_title('b-value Phase Portrait: Seismicity Fingerprints',
             fontsize=14, fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'oklahoma_bvalue_loop.png', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved: figures/oklahoma_bvalue_loop.png')

---
## Bootstrap Confidence Intervals

For each region, compute bootstrap confidence intervals (95%) on the b-value for three temporal periods:
- **Pre-2009**: Before the ramp-up of wastewater injection
- **2009–2015**: Peak injection era
- **Post-2015**: After regulatory volume reductions

In [ ]:
periods = {
    'Pre-2009': (None, '2009-01-01'),
    '2009-2015': ('2009-01-01', '2015-01-01'),
    'Post-2015': ('2015-01-01', None),
}

bootstrap_rows = []

for name, df in regions.items():
    mc = mc_values[name]
    display_name = (name.replace('socal', 'Southern California')
                        .replace('oklahoma', 'Oklahoma')
                        .replace('permian', 'Permian Basin'))
    
    for period_name, (start, end) in periods.items():
        mask = pd.Series(True, index=df.index)
        if start is not None:
            mask &= df['time'] >= pd.Timestamp(start, tz='UTC')
        if end is not None:
            mask &= df['time'] < pd.Timestamp(end, tz='UTC')
        
        subset = df.loc[mask, 'mag'].values
        n_events = len(subset[subset >= mc])
        
        if n_events < 10:
            bootstrap_rows.append({
                'Region': display_name,
                'Period': period_name,
                'N events': n_events,
                'b-value': np.nan,
                '95% CI lower': np.nan,
                '95% CI upper': np.nan,
            })
            continue
        
        b_mean, b_low, b_high = bootstrap_b_value(subset, mc, n_bootstrap=2000)
        bootstrap_rows.append({
            'Region': display_name,
            'Period': period_name,
            'N events': n_events,
            'b-value': round(b_mean, 3),
            '95% CI lower': round(b_low, 3),
            '95% CI upper': round(b_high, 3),
        })

bootstrap_table = pd.DataFrame(bootstrap_rows)
print('Bootstrap b-value Confidence Intervals (95%)')
print('=' * 70)
display(bootstrap_table)

---
## Kolmogorov-Smirnov Tests

We apply two-sample KS tests to compare magnitude distributions across regions and time periods. A low p-value indicates statistically significant differences in the shape of the magnitude distribution, suggesting different generating processes.

In [ ]:
# Use the maximum Mc across all regions for fair comparison
mc_global = max(mc_values.values())

# Define temporal subsets for Oklahoma
ok_df = regions['oklahoma']
ok_pre = ok_df[ok_df['time'] < pd.Timestamp('2009-01-01', tz='UTC')]['mag'].values
ok_during = ok_df[(ok_df['time'] >= pd.Timestamp('2009-01-01', tz='UTC')) &
                  (ok_df['time'] < pd.Timestamp('2015-01-01', tz='UTC'))]['mag'].values
ok_2011_2013 = ok_df[(ok_df['time'] >= pd.Timestamp('2011-01-01', tz='UTC')) &
                     (ok_df['time'] < pd.Timestamp('2014-01-01', tz='UTC'))]['mag'].values

socal_mags = regions['socal']['mag'].values
permian_recent = regions['permian'][
    regions['permian']['time'] >= pd.Timestamp('2020-01-01', tz='UTC')
]['mag'].values

# Define comparisons
comparisons = [
    ('Oklahoma pre-injection vs during-injection', ok_pre, ok_during),
    ('Oklahoma during-injection vs SoCal (all)', ok_during, socal_mags),
    ('Permian recent (2020+) vs Oklahoma 2011-2013', permian_recent, ok_2011_2013),
]

ks_rows = []
for label, mags1, mags2 in comparisons:
    try:
        stat, pval = ks_test_magnitudes(mags1, mags2, mc_global)
        ks_rows.append({
            'Comparison': label,
            'N1': len(mags1[mags1 >= mc_global]),
            'N2': len(mags2[mags2 >= mc_global]),
            'KS statistic': round(stat, 4),
            'p-value': f'{pval:.2e}',
            'Significant (p<0.05)': 'Yes' if pval < 0.05 else 'No',
        })
    except ValueError as e:
        ks_rows.append({
            'Comparison': label,
            'N1': len(mags1[mags1 >= mc_global]),
            'N2': len(mags2[mags2 >= mc_global]),
            'KS statistic': np.nan,
            'p-value': str(e),
            'Significant (p<0.05)': 'N/A',
        })

ks_table = pd.DataFrame(ks_rows)
print('Kolmogorov-Smirnov Tests on Magnitude Distributions')
print('=' * 70)
display(ks_table)

---
## Key Findings

The **b-value phase portrait** reveals fundamentally different dynamical signatures across the three regions:

1. **Oklahoma traces a hysteresis loop.** As wastewater injection ramped up after 2009, Oklahoma's seismicity moved into high-rate, high-b territory — a hallmark of induced seismicity. After regulatory action in 2015, the trajectory began retreating, but along a different path than the outbound journey. This hysteresis suggests that fault systems do not simply "reset" when injection ceases; the stress perturbation leaves a lasting imprint.

2. **Southern California orbits a stable attractor.** SoCal's trajectory remains confined to a compact region near $b \approx 1.0$, fluctuating around a tectonic equilibrium. This bounded behavior is the expected signature of natural, tectonic seismicity and serves as our reference baseline.

3. **The Permian Basin is entering Oklahoma's historical trajectory.** The Permian Basin's current position in phase space shows it drifting toward higher rates and elevated b-values — a pattern that mirrors Oklahoma circa 2011-2013, the early acceleration phase. This is an early-warning signal that the Permian Basin may be on a similar induced seismicity trajectory.

These phase portraits provide a powerful, low-dimensional summary of complex seismicity dynamics. They compress thousands of earthquake records into a trajectory that can be monitored in near-real-time, offering regulators and operators an intuitive visual tool for tracking the emergence of induced seismicity.